In [5]:
import pandas as pd
import seaborn as sns
sns.set()
import mlflow
import subprocess

from pathlib import Path

from qrt.data import load_tr_data
from qrt.splitters import make_date_folds
from qrt.train import train_and_predict_cv, predict_positive_probability

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [6]:
X_tr, y_tr = load_tr_data(Path("../data/raw"))

In [7]:
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("qrt-aapf")

cv_params = {"n_splits": 8, "seed": 42}
folds = make_date_folds(X_tr, **cv_params)

X = X_tr.drop(columns=["TS", "GROUP", "ALLOCATION"])
y = (y_tr["target"] > 0).astype(int)

model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", RobustScaler()),
        ("model", LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42
        )),
    ]
)


In [8]:
root = Path("..").resolve()

with mlflow.start_run(run_name="logreg_missing_indicators") as run:
    # Record the experiment configuration
    mlflow.set_tags({
        "model": "LogisticRegression",
        "reason": "Preserve missingness information after median imputation",
        "git_commit": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=root, text=True
        ).strip(),
    })

    mlflow.log_params(model.get_params())
    mlflow.log_params({
        "cv_n_splits": cv_params["n_splits"],
        "cv_seed": cv_params["seed"],
        "prediction_threshold": 0.5,
    })

    mlflow.log_dict({
        "features": X.columns.tolist(),
        "target": "target > 0",
        "training_files": ["X_train.csv", "y_train.csv"],
        "cv_method": "KFold on shuffled unique TS",
    }, "config.json")

    # Save the source files, including uncommitted changes
    for path in (root / "src/qrt").glob("*.py"):
        mlflow.log_artifact(str(path), artifact_path="source/qrt")

    for name in (
        "pyproject.toml",
        "uv.lock",
        "notebooks/02_models.ipynb",
    ):
        mlflow.log_artifact(str(root / name), artifact_path="source")

    # Train and evaluate
    oof = train_and_predict_cv(model, X, y, folds, predict_positive_probability)
    assert oof.notna().all()

    y_pred = (oof >= 0.5).astype(int)
    scores = {"accuracy": accuracy_score(y, y_pred)}
    for fold in sorted(folds.unique()):
        rows = folds == fold
        scores[f"accuracy_fold_{fold}"] = accuracy_score(y[rows], y_pred[rows])

    mlflow.log_metrics(scores)

    # Save predictions and the exact fold assignments
    predictions = pd.DataFrame({
        "fold": folds,
        "target": y,
        "prediction": oof,
    })
    mlflow.log_text(predictions.to_csv(), "oof.csv")

pd.Series(scores)

🏃 View run logreg_missing_indicators at: http://127.0.0.1:5001/#/experiments/2/runs/4d5476868e6941ea89e5d0fc124ffcb8
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


accuracy           0.519989
accuracy_fold_0    0.520031
accuracy_fold_1    0.520715
accuracy_fold_2    0.517652
accuracy_fold_3    0.521417
accuracy_fold_4    0.508863
accuracy_fold_5    0.523173
accuracy_fold_6    0.523297
accuracy_fold_7    0.525555
dtype: float64